# MEAQ-T Coating: Quickstart Tutorial

This notebook demonstrates the full MEAQ-T workflow:
1. Composition screening (high-entropy alloys)
2. Moire transport characterization
3. Pulse-driven magnetic switching
4. Multi-objective coating stack optimization
5. Results visualization

**Duration**: ~5–10 minutes

## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add src to path if running from notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from meaqt import (
    screen_compositions,
    scan_moire_response,
    simulate_pulse_switching,
    optimize_stacks,
    save_moire_heatmap,
    save_pareto_scatter,
)

print('✓ MEAQ-T modules imported successfully')

## Step 1: High-Entropy Composition Screening

Search the compositional space for stable multicomponent alloys using entropy and thermal proxies.

In [ ]:
# Screen 4-element HEA compositions
comp_df = screen_compositions(n_elements=4, min_entropy_r=1.5, top_k=15)

print(f"Found {len(comp_df)} compositions (top 15)")
print("\nTop 5 by score:")
print(comp_df[['composition', 'S_mix_over_R', 'radius_mismatch', 'avg_tm_k', 'score']].head(5))

**Interpretation:**
- **S_mix_over_R**: Configurational entropy relative to gas constant. Higher is more stabilizing.
- **radius_mismatch**: Atomic size disorder. Moderate values (0.1–0.2) reduce lattice thermal conductivity.
- **avg_tm_k**: Average melting point. Higher ⟹ better high-temperature performance.
- **score**: Weighted combination of all descriptors; higher is better.

## Step 2: Moire Transport Characterization

Compute Seebeck coefficient, conductivity, and power factor as functions of twist angle and displacement field.

In [ ]:
# Scan moire response over twist angle, displacement field, and temperature
theta_grid = np.linspace(0.5, 2.5, 5)
displacement_grid = np.linspace(-0.3, 0.3, 5)
temp_grid = np.array([50.0])  # Single temperature for speed

moire_df = scan_moire_response(theta_grid, displacement_grid, temp_grid, filling=0.0)

print(f"Scanned {len(moire_df)} parameter points")
print("\nTop 5 by power factor:")
print(moire_df[['theta_deg', 'displacement_vnm', 'seebeck_uV_per_K', 'power_factor_arb']].nlargest(5, 'power_factor_arb'))

## Step 3: Pulse-Driven Magnetic Switching

Simulate ultrafast optical pulse switching magnetization via the inverse Faraday effect (IFE).

In [ ]:
# Simulate a single pulse scenario
pulse_result = simulate_pulse_switching(
    duration_ps=20.0,
    pulse_center_ps=5.0,
    pulse_width_ps=0.8,
    pulse_amp_t=-1.2,
    alpha=0.05,
)

print(f"Switched: {pulse_result['switched']}")
print(f"Final mz: {pulse_result['m'][-1, 2]:.4f}")
print(f"Trajectory shape: {pulse_result['m'].shape}")

## Step 4: Coating Stack Optimization

Combine protective, interlayer, moire, and substrate options into a multi-objective score.

In [ ]:
# Run optimization with preset weights (balanced)
coatings_df = optimize_stacks(
    preset='balanced',
    top_k=10,
    score_mode='raw',
)

print(f"\nTop 10 coating stacks:")
print(coatings_df[['protective', 'interlayer', 'moire', 'substrate', 'score']].head(10))

print(f"\nBest stack (rank 1):")
best = coatings_df.iloc[0]
print(f"  Protective: {best['protective']}")
print(f"  Interlayer: {best['interlayer']}")
print(f"  Moire: {best['moire']}")
print(f"  Substrate: {best['substrate']}")
print(f"  Score: {best['score']:.4f}")

## Step 5: Visualization

Generate publication-ready plots of key results.

In [ ]:
# Create output directory
out_dir = Path('figures')
out_dir.mkdir(exist_ok=True)

# Plot 1: Moire heatmap (Seebeck coefficient)
save_moire_heatmap(
    moire_df,
    value_col='seebeck_uV_per_K',
    temp_k=50.0,
    out_path=str(out_dir / 'seebeck_heatmap.png')
)
print(f"✓ Saved {out_dir / 'seebeck_heatmap.png'}")

# Plot 2: Coating Pareto scatter
save_pareto_scatter(
    coatings_df,
    out_path=str(out_dir / 'coating_candidates.png')
)
print(f"✓ Saved {out_dir / 'coating_candidates.png'}")

## Next Steps

1. **Refine compositions**: Screen more elements or use different entropy thresholds.
2. **Expand parameter scans**: Use finer grids or additional temperatures/fillings.
3. **Pulse phase diagrams**: Vary pulse width and amplitude to map switching landscape.
4. **Constraint filtering**: Add hard constraints (e.g., max thermal conductivity).
5. **DFT calibration**: Use parsed DFT data to calibrate band structure models.

See the other notebooks for advanced workflows!